# Module 04: Feature Engineering, Extraction, & Dimensionality Reduction


## Module outline

* **Step 1: Classical Feature Construction**
  * Mathematical combinations: Sums, differences, ratios, and products.
  * Interaction features: `PolynomialFeatures` generation to capture non-linear interactions.
  * Date/Time Engineering: Cyclic feature transformation using Sine and Cosine mappings for time, days, and months.
  * **Discretization / Binning:** Converting continuous features to discrete categories via Uniform (equal-width), Quantile (equal-frequency), and K-Means clustering-based binning.
* **Step 2: Classical Feature Extraction**
  * **Text Processing (Deterministic & Statistical):** Tokenization, Stopword removal, Stemming (Porter, Snowball) vs. Lemmatization (WordNet), Bag-of-Words (CountVectorizer), N-grams (bi-grams, tri-grams), TF-IDF vectorization, and Feature Hashing.
  * **Image Processing (Hand-Crafted Spatial Descriptors):** Image kernels, Gaussian blur, Sobel/Canny Edge Detection, Haralick Texture features, Histograms of Oriented Gradients (HOG), and Scale-Invariant Feature Transform (SIFT).
  * **Signal & Time-Series Extraction:** Lag features, lead features, rolling/window statistics (rolling mean, rolling std, expanding window), and Fourier Transforms for frequency-domain analysis.
* **Step 3: Feature Selection Methodologies**
  * Filter Methods: Constant/Quasi-constant feature removal, Variance Thresholding, Pearson's Correlation filtering, Chi-Square test selection, ANOVA F-value selection, and Mutual Information score filtering.
  * Wrapper Methods: Forward Feature Selection, Backward Feature Elimination, Exhaustive Feature Selection, and Recursive Feature Elimination (RFE / RFECV).
  * Embedded Methods: Regularization-driven selection (L1 penalty), tree-based feature importance, and Permutation Importance.
* **Step 4: Dimensionality Reduction (PCA Deep Dive)**
  * **Mathematical PCA Execution:**
    1. Mean centering and standardization of the design matrix $X$.
    2. Covariance Matrix calculation: $\Sigma = \frac{1}{n-1} X^T X$.
    3. Eigendecomposition of the covariance matrix: $\Sigma V = \lambda V$ (extracting Eigenvalues $\lambda$ and Eigenvectors $V$).
    4. Sorting eigenvalues in descending order to measure explained variance.
    5. Constructing the projection matrix $W$ using the top $k$ eigenvectors.
    6. Transforming original features to the new coordinate space: $Z = XW$.
  * Supervised Dimensionality Reduction: Linear Discriminant Analysis (LDA) for maximizing class separation.
  * Non-Linear Dimensionality Reduction: Kernel PCA (using RBF/Polynomial kernels).
  * Matrix Factorization: Non-Negative Matrix Factorization (NMF) and Singular Value Decomposition (SVD).
  * Manifold Learning: t-SNE (t-Distributed Stochastic Neighbor Embedding) and UMAP (Uniform Manifold Approximation and Projection) for high-dimensional structures.


## Example Dataset


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

# Let's define our exact 10-employee dataset in Python to display it cleanly
data = {
    'Employee ID': ['Emp 1', 'Emp 2', 'Emp 3', 'Emp 4', 'Emp 5', 'Emp 6', 'Emp 7', 'Emp 8', 'Emp 9', 'Emp 10'],
    'Department': ['Sales', 'R&D', 'HR', 'R&D', 'Sales', 'R&D', 'HR', 'Sales', 'R&D', 'Sales'],
    'Performance Rating': ['Low', 'Medium', 'Medium', 'High', 'High', 'Low', 'High', 'Medium', 'High', 'Medium'],
    'Years at Company': [1, 3, 2, 5, 4, 1, 8, 3, 10, 3],
    'Monthly Salary': [3000, 4500, 4000, 6000, 5500, 3200, 7000, 4800, 12000, 5000],
    'Attrition (Left?)': ['Yes (1)', 'No (0)', 'No (0)', 'No (0)', 'No (0)', 'Yes (1)', 'No (0)', 'No (0)', 'No (0)', 'Yes (1)']
}

df = pd.DataFrame(data)
display(df)


,Employee ID,Department,Performance Rating,Years at Company,Monthly Salary,Attrition (Left?)
0,Emp 1,Sales,Low,1,3000,Yes (1)
1,Emp 2,R&D,Medium,3,4500,No (0)
2,Emp 3,HR,Medium,2,4000,No (0)
3,Emp 4,R&D,High,5,6000,No (0)
4,Emp 5,Sales,High,4,5500,No (0)
5,Emp 6,R&D,Low,1,3200,Yes (1)
6,Emp 7,HR,High,8,7000,No (0)
7,Emp 8,Sales,Medium,3,4800,No (0)
8,Emp 9,R&D,High,10,12000,No (0)
9,Emp 10,Sales,Medium,3,5000,Yes (1)


## Module 4: Feature Engineering, Extraction, and Dimensionality Reduction

This module answers one main question:

```text
How do we turn raw columns into better model inputs?
```

There are four related ideas:

- **Feature construction:** create new columns from existing tabular columns.
- **Feature extraction:** convert raw data like text, images, and time signals into columns.
- **Feature selection:** keep useful columns and remove weak or repeated columns.
- **Dimensionality reduction:** combine many columns into fewer new columns.

The goal is not to create more columns blindly. The goal is to create useful information and reduce noise.


### Module 4: Step 1 - Classical Feature Construction

- **Definition:** Feature construction means creating new columns from columns we already have.
- **Where we use it:** Tabular data such as salary, tenure, department, dates, counts, prices, and customer activity.
- **Why we use it:** Sometimes the useful pattern is not directly visible in one original column.

#### 1. Mathematical Combinations

These are simple arithmetic features.

| Type | Example | Meaning |
|---|---|---|
| Sum | `total_score = test1 + test2` | total amount |
| Difference | `salary_gap = salary - dept_avg_salary` | above/below group average |
| Ratio | `salary_per_year = salary / years` | value relative to another value |
| Product | `salary_x_years = salary * years` | combined effect |

**Important:** Ratios need a nonzero denominator. If `years = 0`, then `salary / years` is not valid.

#### 2. Interaction Features

An interaction feature checks whether two columns together carry meaning.

Example:

```text
Years at Company x Monthly Salary
```

This can help when salary has a different meaning for new employees and experienced employees.

#### 3. Polynomial Features

Polynomial features create powers and products of numerical columns.

Example with two columns `x` and `y`:

```text
x, y, x*y, x^2, y^2
```

- **Where we use it:** Simple models that need help learning curved or combined patterns.
- **Why we use it:** A linear model can become more flexible when we add these extra columns.
- **Careful:** Polynomial features can create too many columns quickly.

#### 4. Date/Time Engineering

Dates should usually be split into useful parts.

Examples:

```text
Joining Date -> joining month, joining year, day of week
```

For repeating cycles, plain numbers can be misleading. Month `12` and month `1` are close in real life, but numerically they look far apart.

Cyclic encoding fixes this using sine and cosine:

```text
month_sin = sin(2*pi*month/12)
month_cos = cos(2*pi*month/12)
```

This keeps the circular nature of months, days, and hours.

#### 5. Discretization / Binning

Binning means converting continuous numbers into groups.

Example:

```text
Years at Company -> New, Growing, Senior
```

Common binning methods:

- **Uniform / equal-width binning:** each bin covers the same numerical range.
- **Quantile / equal-frequency binning:** each bin has a similar number of rows.
- **K-Means binning:** bins are formed around clusters in the values.

Binning can make interpretation easier, but it loses exact numerical detail.


In [2]:
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer
import pandas as pd
import numpy as np

# Create a copy to construct custom features
df_features = df.copy()

# Artificial joining month only for teaching cyclic date/time features
df_features['Joining_Month'] = [1, 3, 4, 6, 7, 9, 10, 11, 12, 2]

# 1. Mathematical combinations
df_features['Salary_Per_Year'] = df_features['Monthly Salary'] / df_features['Years at Company']
department_avg_salary = df_features.groupby('Department')['Monthly Salary'].transform('mean')
df_features['Salary_Gap_From_Department_Avg'] = df_features['Monthly Salary'] - department_avg_salary
df_features['Salary_x_Tenure'] = df_features['Monthly Salary'] * df_features['Years at Company']

# 2. Cyclic date/time features for month
month_angle = 2 * np.pi * df_features['Joining_Month'] / 12
df_features['Joining_Month_Sin'] = np.sin(month_angle).round(3)
df_features['Joining_Month_Cos'] = np.cos(month_angle).round(3)

# 3. Custom business binning
df_features['Tenure_Group'] = pd.cut(
    df_features['Years at Company'],
    bins=[0, 2, 5, np.inf],
    labels=['New', 'Growing', 'Senior']
)

# 4. Uniform, quantile, and k-means binning examples
bin_input = df_features[['Years at Company']]
for strategy in ['uniform', 'quantile', 'kmeans']:
    binner = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy=strategy, subsample=None)
    df_features[f'Tenure_Bin_{strategy}'] = binner.fit_transform(bin_input).astype(int)

print('Constructed feature examples:')
display(df_features[[
    'Employee ID',
    'Department',
    'Years at Company',
    'Monthly Salary',
    'Salary_Per_Year',
    'Salary_Gap_From_Department_Avg',
    'Salary_x_Tenure',
    'Joining_Month',
    'Joining_Month_Sin',
    'Joining_Month_Cos',
    'Tenure_Group',
    'Tenure_Bin_uniform',
    'Tenure_Bin_quantile',
    'Tenure_Bin_kmeans'
]])

# 5. Polynomial/interaction feature example
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
interaction_features = poly.fit_transform(df_features[['Years at Company', 'Monthly Salary']])
interaction_df = pd.DataFrame(
    interaction_features,
    columns=poly.get_feature_names_out(['Years at Company', 'Monthly Salary'])
)

print('\nInteraction feature example:')
display(interaction_df.head())


C:\Users\91832\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\91832\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 282, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


Constructed feature examples:


,Employee ID,Department,Years at Company,Monthly Salary,Salary_Per_Year,Salary_Gap_From_Department_Avg,Salary_x_Tenure,Joining_Month,Joining_Month_Sin,Joining_Month_Cos,Tenure_Group,Tenure_Bin_uniform,Tenure_Bin_quantile,Tenure_Bin_kmeans
0,Emp 1,Sales,1,3000,3000.000000,-1575.0,3000,1,0.500,0.866,New,0,0,0
1,Emp 2,R&D,3,4500,1500.000000,-1925.0,13500,3,1.000,0.000,Growing,0,1,0
2,Emp 3,HR,2,4000,2000.000000,-1500.0,8000,4,0.866,-0.500,New,0,0,0
3,Emp 4,R&D,5,6000,1200.000000,-425.0,30000,6,0.000,-1.000,Growing,1,2,1
4,Emp 5,Sales,4,5500,1375.000000,925.0,22000,7,-0.500,-0.866,Growing,1,1,1
5,Emp 6,R&D,1,3200,3200.000000,-3225.0,3200,9,-1.000,-0.000,New,0,0,0
6,Emp 7,HR,8,7000,875.000000,1500.0,56000,10,-0.866,0.500,Senior,2,2,2
7,Emp 8,Sales,3,4800,1600.000000,225.0,14400,11,-0.500,0.866,Growing,0,1,0
8,Emp 9,R&D,10,12000,1200.000000,5575.0,120000,12,-0.000,1.000,Senior,2,2,2
9,Emp 10,Sales,3,5000,1666.666667,425.0,15000,2,0.866,0.500,Growing,0,1,0



Interaction feature example:


,Years at Company,Monthly Salary,Years at Company Monthly Salary
0,1.0,3000.0,3000.0
1,3.0,4500.0,13500.0
2,2.0,4000.0,8000.0
3,5.0,6000.0,30000.0
4,4.0,5500.0,22000.0


### Module 4: Step 2 - Classical Feature Extraction

- **Definition:** Feature extraction means converting raw information into numerical features.
- **Where we use it:** Text, images, audio, signals, and time-series data.
- **Why we use it:** Raw text or images are not directly useful for most classical ML models.

#### 1. Text Processing

Text processing converts sentences into numbers.

| Concept | Simple meaning |
|---|---|
| Tokenization | split text into words or pieces |
| Stopword removal | remove very common words like `the`, `is`, `and` |
| Stemming | cut words to a rough root, such as `leaving` -> `leav` |
| Lemmatization | convert words to dictionary form, such as `leaving` -> `leave` |
| Bag-of-Words | count which words appear |
| N-grams | count word groups, such as two-word phrases |
| TF-IDF | give more weight to words that are important in one document but not common everywhere |
| Feature Hashing | convert text into fixed-size columns without storing a full vocabulary |

**Bag-of-Words vs TF-IDF:**

- Bag-of-Words asks: how many times did this word appear?
- TF-IDF asks: is this word important in this document compared with all documents?

#### 2. Image Processing

Image feature extraction turns pixels into useful measurements.

| Concept | Simple meaning |
|---|---|
| Image kernel | small filter moved across an image |
| Gaussian blur | smooth the image and reduce noise |
| Sobel / Canny edges | find boundaries and sharp changes |
| Haralick texture | describe repeated texture patterns |
| HOG | summarize edge directions in local regions |
| SIFT | find distinctive points that can be matched across images |

These are called hand-crafted image features because humans designed the feature rules.

#### 3. Signal and Time-Series Extraction

Time-series data has order, so past values can become useful features.

| Concept | Simple meaning |
|---|---|
| Lag feature | previous value, such as `x(t-1)` |
| Lead feature | future value, useful for analysis but dangerous for prediction |
| Rolling mean | recent average over a window |
| Rolling standard deviation | recent variability over a window |
| Expanding window | summary from the start up to the current time |
| Fourier transform | finds repeating frequency patterns |

**Leakage warning:** For prediction, do not use future values. A lead feature can leak the answer if used incorrectly.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Text feature extraction example
text_feedback = [
    'I am leaving because of low salary limits',
    'No intention to leave the company',
    'The company culture is excellent and supportive',
    'The R&D team provides excellent growth opportunities',
    'Sales targets are high but salary is competitive',
    'Leaving because of stress and low growth options',
    'No intention to leave stable job security',
    'Competitive salary is keeping me here',
    'Extremely happy with my current role',
    'Leaving the company due to long commute'
]

tfidf = TfidfVectorizer(stop_words='english', max_features=6, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(text_feedback)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

print('Text converted to TF-IDF features:')
display(tfidf_df.head())

# Time-series feature extraction example using a simple salary sequence
salary_series = pd.Series(df['Monthly Salary'].values, name='Monthly Salary')
time_features = pd.DataFrame({
    'Salary': salary_series,
    'Lag_1': salary_series.shift(1),
    'Lead_1_for_analysis_only': salary_series.shift(-1),
    'Rolling_Mean_3': salary_series.rolling(window=3).mean(),
    'Rolling_Std_3': salary_series.rolling(window=3).std(),
    'Expanding_Mean': salary_series.expanding().mean()
})

print('\nSimple time-series extracted features:')
display(time_features)


Text converted to TF-IDF features:


,company,growth,intention leave,leave,leaving,salary
0,0.000000,0.0,0.000000,0.000000,0.707107,0.707107
1,0.526101,0.0,0.601339,0.601339,0.000000,0.000000
2,1.000000,0.0,0.000000,0.000000,0.000000,0.000000
3,0.000000,1.0,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.0,0.000000,0.000000,0.000000,1.000000



Simple time-series extracted features:


,Salary,Lag_1,Lead_1_for_analysis_only,Rolling_Mean_3,Rolling_Std_3,Expanding_Mean
0,3000,NaN,4500.0,NaN,NaN,3000.000000
1,4500,3000.0,4000.0,NaN,NaN,3750.000000
2,4000,4500.0,6000.0,3833.333333,763.762616,3833.333333
3,6000,4000.0,5500.0,4833.333333,1040.833000,4375.000000
4,5500,6000.0,3200.0,5166.666667,1040.833000,4600.000000
5,3200,5500.0,7000.0,4900.000000,1493.318452,4366.666667
6,7000,3200.0,4800.0,5233.333333,1913.983629,4742.857143
7,4800,7000.0,12000.0,5000.000000,1907.878403,4750.000000
8,12000,4800.0,5000.0,7933.333333,3689.625094,5555.555556
9,5000,12000.0,NaN,7266.666667,4100.406484,5500.000000


### Module 4: Step 3 - Feature Selection Methodologies

- **Definition:** Feature selection means choosing useful input columns and removing weak, repeated, or harmful columns.
- **Where we use it:** Datasets with many features, duplicated information, constant columns, or noisy columns.
- **Why we use it:** A smaller useful feature set can make the model simpler, faster, and easier to explain.

Feature selection keeps original columns. It does not create new combined columns.

#### 1. Filter Methods

Filter methods select features using statistics before fitting the final model.

| Method | Simple meaning |
|---|---|
| Constant feature removal | remove columns with only one value |
| Quasi-constant removal | remove columns that almost never change |
| Variance threshold | remove low-variation columns |
| Pearson correlation filtering | remove columns that are nearly duplicates of each other |
| Chi-square test | check relationship between nonnegative features and class labels |
| ANOVA F-value | check whether a numerical feature separates target classes |
| Mutual information | measure dependency, including some non-linear patterns |

#### 2. Wrapper Methods

Wrapper methods repeatedly train a model while trying different feature subsets.

| Method | Simple meaning |
|---|---|
| Forward selection | start with no features, add one feature at a time |
| Backward elimination | start with all features, remove weak features one at a time |
| Exhaustive selection | try every possible feature subset; accurate but expensive |
| RFE | repeatedly remove the least useful feature |
| RFECV | RFE with cross-validation to choose how many features to keep |

Wrapper methods can be strong, but they are slower because they train many models.

#### 3. Embedded Methods

Embedded methods select features during model training.

| Method | Simple meaning |
|---|---|
| L1 / Lasso | pushes weak coefficients to zero |
| Tree-based importance | features used for useful tree splits get importance |
| Permutation importance | shuffle one feature and see how much model performance drops |

#### Important Rule

Feature selection must be fitted only on training data. If the test set is used during feature selection, the evaluation becomes biased.


In [4]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Prepare structured features from our 10-employee dataset
X_select = df[['Years at Company', 'Monthly Salary']].copy()
X_select['Salary_Per_Year'] = df['Monthly Salary'] / df['Years at Company']
X_select['Constant_Feature'] = 1

y_binary = df['Attrition (Left?)'].apply(lambda x: 1 if 'Yes' in x else 0)

print('Original feature matrix:')
display(X_select.head())

# 1. Filter method: remove zero-variance feature
selector_var = VarianceThreshold(threshold=0.1)
X_variance_filtered = selector_var.fit_transform(X_select)

print('\n1. Filter method: VarianceThreshold removed constant features')
display(pd.DataFrame(X_variance_filtered, columns=selector_var.get_feature_names_out()))

# 2. Filter method: ANOVA F-value score
anova_selector = SelectKBest(score_func=f_classif, k=2)
anova_selector.fit(X_select.drop(columns=['Constant_Feature']), y_binary)
anova_scores = pd.DataFrame({
    'Feature': anova_selector.feature_names_in_,
    'ANOVA_F_Score': anova_selector.scores_.round(3),
    'Selected?': anova_selector.get_support()
})

print('\n2. Filter method: ANOVA F-value scores')
display(anova_scores)

# 3. Wrapper method: RFE using logistic regression
X_scaled = StandardScaler().fit_transform(X_select.drop(columns=['Constant_Feature']))
estimator = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator, n_features_to_select=1)
rfe.fit(X_scaled, y_binary)

ranking_df = pd.DataFrame({
    'Feature': X_select.drop(columns=['Constant_Feature']).columns,
    'RFE Rank': rfe.ranking_,
    'Selected?': rfe.support_
})

print('\n3. Wrapper method: RFE ranking')
display(ranking_df)

# 4. Embedded method: L1 logistic regression can shrink weak coefficients toward zero
l1_model = LogisticRegression(penalty='l1', solver='liblinear', C=0.5, random_state=42)
l1_model.fit(X_scaled, y_binary)

l1_df = pd.DataFrame({
    'Feature': X_select.drop(columns=['Constant_Feature']).columns,
    'L1_Coefficient': l1_model.coef_[0].round(4)
})

print('\n4. Embedded method: L1 coefficients')
display(l1_df)


Original feature matrix:

,Years at Company,Monthly Salary,Salary_Per_Year,Constant_Feature
0,1,3000,3000.0,1
1,3,4500,1500.0,1
2,2,4000,2000.0,1
3,5,6000,1200.0,1
4,4,5500,1375.0,1



1. Filter method: VarianceThreshold removed constant features


,Years at Company,Monthly Salary,Salary_Per_Year
0,1.0,3000.0,3000.000000
1,3.0,4500.0,1500.000000
2,2.0,4000.0,2000.000000
3,5.0,6000.0,1200.000000
4,4.0,5500.0,1375.000000
5,1.0,3200.0,3200.000000
6,8.0,7000.0,875.000000
7,3.0,4800.0,1600.000000
8,10.0,12000.0,1200.000000
9,3.0,5000.0,1666.666667



2. Filter method: ANOVA F-value scores


,Feature,ANOVA_F_Score,Selected?
0,Years at Company,3.415,True
1,Monthly Salary,2.281,False
2,Salary_Per_Year,11.779,True



3. Wrapper method: RFE ranking


,Feature,RFE Rank,Selected?
0,Years at Company,2,False
1,Monthly Salary,3,False
2,Salary_Per_Year,1,True



4. Embedded method: L1 coefficients


,Feature,L1_Coefficient
0,Years at Company,0.0000
1,Monthly Salary,0.0000
2,Salary_Per_Year,0.6729


### Module 4: Step 4 - Dimensionality Reduction

- **Definition:** Dimensionality reduction means representing many columns using fewer columns.
- **Where we use it:** High-dimensional numerical data, correlated features, text vectors, image features, and visualization.
- **Why we use it:** Fewer dimensions can reduce noise, improve speed, and make visualization easier.

Feature selection and dimensionality reduction are different:

```text
Feature selection: keep some original columns.
Dimensionality reduction: create new compressed columns.
```

#### PCA Deep Dive

PCA means Principal Component Analysis.

Simple meaning:

```text
PCA finds the directions where the data varies the most.
The strongest direction becomes PC1.
The next strongest direction becomes PC2.
```

Mathematical workflow:

1. **Mean center / standardize the data matrix `X`.**
   - Centering subtracts the mean.
   - Standardization also divides by standard deviation.
2. **Compute covariance matrix.**
   - Covariance shows how features move together.
   - Formula: `Sigma = (1 / (n - 1)) X^T X`
3. **Find eigenvectors and eigenvalues.**
   - Eigenvectors are directions.
   - Eigenvalues tell how much variance exists in those directions.
4. **Sort eigenvalues from largest to smallest.**
   - Largest eigenvalue means most variation.
5. **Choose the top `k` eigenvectors.**
   - These become the projection matrix `W`.
6. **Transform the data.**
   - Formula: `Z = XW`
   - `Z` is the new compressed representation.

#### Other Dimensionality Reduction Methods

| Method | Type | Simple meaning |
|---|---|---|
| LDA | supervised | creates directions that separate classes |
| Kernel PCA | non-linear | PCA idea with curved/non-linear structure |
| NMF | matrix factorization | breaks nonnegative data into additive parts |
| SVD | matrix factorization | decomposes a matrix into important hidden directions |
| t-SNE | manifold learning | visualizes high-dimensional data in 2D/3D |
| UMAP | manifold learning | visualizes or compresses high-dimensional structure |

#### When to Use PCA

Use PCA when:

- many numerical features are correlated
- you need fewer columns
- speed or compression matters
- interpretability of each original feature is less important

Be careful when:

- original feature meaning is important
- features are not scaled correctly
- you expect PCA to automatically improve prediction accuracy
- you are using t-SNE/UMAP as if they were normal production preprocessing methods


In [5]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Prepare features to compress
X_raw = df[['Years at Company', 'Monthly Salary']]

# Standardize because the two columns use very different scales
scaler = StandardScaler()
X_std = scaler.fit_transform(X_raw)

# Run PCA with both components so we can see all explained variance
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f'PC{i+1}' for i in range(X_pca.shape[1])]
)

print('Explained variance ratio per component:')
for idx, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{idx+1}: {var*100:.2f}%')

print('\nOriginal standardized features vs PCA coordinates:')
pca_compare = pd.DataFrame({
    'Std Years': X_std[:, 0],
    'Std Salary': X_std[:, 1],
    f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)': pca_df['PC1'],
    f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)': pca_df['PC2']
})
display(pca_compare)


Explained variance ratio per component:


  PC1: 97.47%
  PC2: 2.53%

Original standardized features vs PCA coordinates:


,Std Years,Std Salary,PC1 (97.47% variance),PC2 (2.53% variance)
0,-1.074172,-1.018248,-1.479564e+00,3.954476e-02
1,-0.358057,-0.407299,-5.411887e-01,-3.481907e-02
2,-0.716115,-0.610949,-9.383755e-01,7.436382e-02
3,0.358057,0.203650,3.971868e-01,-1.091829e-01
4,0.000000,0.000000,1.962616e-17,-3.925231e-18
5,-1.074172,-0.936788,-1.421964e+00,9.714554e-02
6,1.432230,0.610949,1.444745e+00,-5.807335e-01
7,-0.358057,-0.285109,-4.547876e-01,5.158211e-02
8,2.148345,2.647444,3.391134e+00,3.529163e-01
9,-0.358057,-0.203650,-3.971868e-01,1.091829e-01
